In [1]:
## Libraries
import numpy as np
import pandas as pd
import librosa
from pathlib import Path


In [2]:
## Parameters
SAMPLING_RATE=16000

## No of frequency bins
## groups the similiar freq sampling rate/n_fft no. of frequencies in each bin 31.25 here
N_FFT=512

## shifts the frame by hop_length data in the audio array
HOP_LENGTH=256
WIN_LEN=512

## Window of different frames from stft,
WINDOW=20
STRIDE=10

In [3]:
## Dataset loading
clean_folder='16k-LP7'
noisy_folder='noisy_dataset'

## Listing all the files in the dataset folder
clean_files=list(Path(clean_folder).rglob('*.wav'))
noisy_files=list(Path(noisy_folder).rglob('*.wav'))


In [4]:
from sklearn.model_selection import train_test_split
train_noisy,test_noisy=train_test_split(noisy_files,test_size=0.3, random_state=110)

In [5]:
train_noisy

[WindowsPath('noisy_dataset/MA01_05_snr5_noisefreesound_community-leaf-blower-77431.wav'),
 WindowsPath('noisy_dataset/MC17_03_snr15_noisefreesound_community-snowblower2-88648.wav'),
 WindowsPath('noisy_dataset/FD22_03_snr0_noisefreesound_community-snowblower2-88648.wav'),
 WindowsPath('noisy_dataset/MK62_08_snr0_noisefreesound_community-industrial-blower-recording-31979.wav'),
 WindowsPath('noisy_dataset/ML72_06_snr5_noisefreesound_community-lawn-mowing-17230.wav'),
 WindowsPath('noisy_dataset/FC14_06_snr5_noisefreesound_community-leaf-blower-77431.wav'),
 WindowsPath('noisy_dataset/FK63_08_snr10_noisefreesound_community-leaf-blower-77431.wav'),
 WindowsPath('noisy_dataset/MD22_07_snr10_noisefreesound_community-leaf-blower-77431.wav'),
 WindowsPath('noisy_dataset/MB09_10_snr0_noisefreesound_community-lawn-mowing-17230.wav'),
 WindowsPath('noisy_dataset/FC18_08_snr10_noisefreesound_community-leaf-blower-77431.wav'),
 WindowsPath('noisy_dataset/CA03_02_snr10_noisefreesound_community-sno

In [6]:
test_noisy

[WindowsPath('noisy_dataset/MF35_03_snr10_noisefreesound_community-leaf-blower-77431.wav'),
 WindowsPath('noisy_dataset/FB07_02_snr0_noisefreesound_community-leaf-blower-77431.wav'),
 WindowsPath('noisy_dataset/CA01_09_snr15_noisefreesound_community-snowblower2-88648.wav'),
 WindowsPath('noisy_dataset/MA03_02_snr5_noisefreesound_community-snowblower2-88648.wav'),
 WindowsPath('noisy_dataset/ML69_03_snr5_noisefreesound_community-leaf-blower-77431.wav'),
 WindowsPath('noisy_dataset/FD20_09_snr5_noisefreesound_community-industrial-blower-recording-31979.wav'),
 WindowsPath('noisy_dataset/FL69_09_snr15_noisefreesound_community-lawn-mowing-17230.wav'),
 WindowsPath('noisy_dataset/MH44_02_snr10_noisefreesound_community-industrial-blower-recording-31979.wav'),
 WindowsPath('noisy_dataset/MK66_07_snr10_noisefreesound_community-industrial-blower-recording-31979.wav'),
 WindowsPath('noisy_dataset/MJ55_09_snr5_noisefreesound_community-industrial-blower-recording-31979.wav'),
 WindowsPath('noisy_d

In [7]:
## Dictionary for fast cleanup
clean_dict={}

## .stem removes the extension (.wav)
for file in clean_files:
    clean_dict[file.stem]=file

X=[]
Y=[]

for noisy_path in train_noisy:

    ## base speech name
    base_name=noisy_path.stem.split('_snr')[0]

    if base_name not in clean_dict:
        continue

    clean_path=clean_dict[base_name]

    ## loading audio
    clean,_=librosa.load(clean_path,sr=SAMPLING_RATE)
    noisy,_=librosa.load(noisy_path,sr=SAMPLING_RATE)

    ##STFT
    ## Gives the excel type matrix with different time frames with energy of each bin
    ##  F1|F2|F3
    ##B1 3|2|4
    ##B2 1|1|7
    ##B3 9|6|3

    ## window='hann' fades the amplitude at the starting and the end of our frame
    clean_stft=librosa.stft(clean, n_fft=N_FFT,hop_length=HOP_LENGTH,win_length=WIN_LEN,window='hann')
    noisy_stft=librosa.stft(noisy, n_fft=N_FFT,hop_length=HOP_LENGTH,win_length=WIN_LEN,window='hann')

    ## Magnitude
    clean_mag=np.abs(clean_stft)
    noisy_mag=np.abs(noisy_stft)

    ## Phase
    clean_phase=np.angle(clean_stft)
    noisy_phase=np.angle(noisy_stft)

    ## Phase difference
    phase_diff=clean_phase-noisy_phase

    ## Phase sensitive mask
    ## the noise bins in the clean mag will be probably empty
    ## but the noise bins will be there 
    ## thus the magnitude ratio will be extremely low in the noisy freq bands
    ## we multiply it with the phase diff the phase diff tells time it is seen first in the frame
    psm=(clean_mag/(noisy_mag+1e-8))*np.cos(phase_diff)

    ## Model input
    ## LOG1P adds 1 with all the values of magnitude since the near to zero mag give infinity on log
    ## this scales magnitudes properly considering low mag and high mags
    log_noisy=np.log1p(noisy_mag)

    ## Frame segmentation
    freq_bins, time_frames=log_noisy.shape


    for start in range(0, time_frames-WINDOW +1,STRIDE):
        end=start+WINDOW
        x_seg=log_noisy[:,start:end]
        y_seg=psm[:,start:end]

        

        X.append(x_seg)
        Y.append(y_seg)
        
print('Total training samples:',len(X))
print('PSM dataset prepared sucessfully')

Total training samples: 13628
PSM dataset prepared sucessfully


In [8]:
## Saving the extracted data
data=np.savez_compressed('stft_train.npz',x=X,y=Y)

In [9]:
## Dictionary for fast cleanup
clean_dict={}

## .stem removes the extension (.wav)
for file in clean_files:
    clean_dict[file.stem]=file

X=[]
Y=[]

for noisy_path in test_noisy:

    ## base speech name
    base_name=noisy_path.stem.split('_snr')[0]

    if base_name not in clean_dict:
        continue

    clean_path=clean_dict[base_name]

    ## loading audio
    clean,_=librosa.load(clean_path,sr=SAMPLING_RATE)
    noisy,_=librosa.load(noisy_path,sr=SAMPLING_RATE)

    ##STFT
    ## Gives the excel type matrix with different time frames with energy of each bin
    ##  F1|F2|F3
    ##B1 3|2|4
    ##B2 1|1|7
    ##B3 9|6|3

    ## window='hann' fades the amplitude at the starting and the end of our frame
    clean_stft=librosa.stft(clean, n_fft=N_FFT,hop_length=HOP_LENGTH,win_length=WIN_LEN,window='hann')
    noisy_stft=librosa.stft(noisy, n_fft=N_FFT,hop_length=HOP_LENGTH,win_length=WIN_LEN,window='hann')

    ## Magnitude
    clean_mag=np.abs(clean_stft)
    noisy_mag=np.abs(noisy_stft)

    ## Phase
    clean_phase=np.angle(clean_stft)
    noisy_phase=np.angle(noisy_stft)

    ## Phase difference
    phase_diff=clean_phase-noisy_phase

    ## Phase sensitive mask
    ## the noise bins in the clean mag will be probably empty
    ## but the noise bins will be there 
    ## thus the magnitude ratio will be extremely low in the noisy freq bands
    ## we multiply it with the phase diff the phase diff tells time it is seen first in the frame
    psm=(clean_mag/(noisy_mag+1e-8))*np.cos(phase_diff)

    ## Model input
    ## LOG1P adds 1 with all the values of magnitude since the near to zero mag give infinity on log
    ## this scales magnitudes properly considering low mag and high mags
    log_noisy=np.log1p(noisy_mag)

    ## Frame segmentation
    freq_bins, time_frames=log_noisy.shape


    for start in range(0, time_frames-WINDOW +1,STRIDE):
        end=start+WINDOW
        x_seg=log_noisy[:,start:end]
        y_seg=psm[:,start:end]

        

        X.append(x_seg)
        Y.append(y_seg)
        
print('Total training samples:',len(X))
print('PSM dataset prepared sucessfully')

Total training samples: 5795
PSM dataset prepared sucessfully


In [10]:
## Convert to numpy
X=np.array(X)
Y=np.array(Y)

In [11]:
## Transpose for RNN
## this reorganizes the 3d matrix of our features
X=np.transpose(X,(0,2,1))
Y=np.transpose(Y,(0,2,1))

In [12]:
print('Dataset Ready')
print('Input shape:',X.shape)
print('Target shape:',Y.shape)

Dataset Ready
Input shape: (5795, 20, 257)
Target shape: (5795, 20, 257)


In [13]:
## Sving the extracted data
data=np.savez_compressed('stft_test.npz',x=X,y=Y)

In [14]:
data=np.load('stft_train.npz')
X=data['x']
Y=data['y']